# Model Controller 

> This notebook contains some example of the Model Controller

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| default_exp model_main

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import os, sys
from transformers import Trainer, TrainingArguments
from datasets import DatasetDict,Dataset
from pathlib import Path
import torch
import gc
from sklearn.metrics import accuracy_score
from functools import partial
import pandas as pd
import numpy as np
from that_nlp_library.utils import *
from that_nlp_library.text_main import TextDataMain

comet_ml is installed but `COMET_API_KEY` is not set.


In [ ]:
#| export
def finetune(lr, # Learning rate
             bs, # Batch size
             wd, # Weight decay
             epochs, # Number of epochs
             ddict, # The HuggingFace datasetdict
             tokenizer,# HuggingFace tokenizer
             o_dir = './tmp_weights', # Directory to save weights
             save_checkpoint=False, # Whether to save weights (checkpoints) to o_dir
             model=None, # NLP model
             model_init=None, # A function to initialize model
             data_collator=None, # HuggingFace data collator
             compute_metrics=None, # A function to compute metric, e.g. `compute_metrics_classification`
             grad_accum_steps=2, # The batch at each step will be divided by this integer and gradient will be accumulated over gradient_accumulation_steps steps.
             lr_scheduler_type='cosine',  # The scheduler type to use. Including: linear, cosine, cosine_with_restarts, polynomial, constant, constant_with_warmup
             no_valid=False, # Whether there is a validation set or not
             seed=42, # Random seed
             report_to='none' # The list of integrations to report the results and logs to. Supported platforms are "azure_ml", "comet_ml", "mlflow", "neptune", "tensorboard","clearml" and "wandb". Use "all" to report to all integrations installed, "none" for no integrations.
            ):
    "The main model training/finetuning function"
    torch.cuda.empty_cache()
    gc.collect()

    seed_everything(seed)
    training_args = TrainingArguments(o_dir, 
                                learning_rate=lr, 
                                warmup_ratio=0.1 if lr_scheduler_type=='cosine' else 0.0, 
                                lr_scheduler_type=lr_scheduler_type, 
                                fp16=True,
                                do_train=True,
                                do_eval= not no_valid,
                                evaluation_strategy="no" if no_valid else "epoch", 
                                save_strategy="epoch" if save_checkpoint else 'no',
                                overwrite_output_dir=True,
                                gradient_accumulation_steps=grad_accum_steps,
                                per_device_train_batch_size=bs, 
                                per_device_eval_batch_size=bs,
                                num_train_epochs=epochs, weight_decay=wd,
                                report_to=report_to,
                                logging_dir=os.path.join(o_dir, 'log') if report_to!='none' else None,
                                logging_steps = len(ddict["train"]) // bs,
                                )

    # instantiate trainer
    trainer = Trainer(
        model=model,
        model_init=model_init if model is None else None,
        args=training_args,
        train_dataset=ddict['train'],#.shard(200, 0),    # Only use subset of the dataset for a quick training. Remove shard for full training
        eval_dataset=ddict['validation'] if not no_valid else None,#.shard(100, 0), # Only use subset of the dataset for a quick training. Remove shard for full training
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    
    
    trainer.train()
    return trainer

In [ ]:
show_doc(finetune)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/IPython/core/formatters.py:223  │
│ in catch_format_error                                                                            │
│                                                                                                  │
│    220 def catch_format_error(method, self, *args, **kwargs):                                    │
│    221 │   """show traceback on failed format call"""                                            │
│    222 │   try:                                                                                  │
│ ❱  223 │   │   r = method(self, *args, **kwargs)                                                 │
│    224 │   except NotImplementedError:                                                           │
│    225 │   │   # don't warn on NotImplementedErrors                                              │
│    226 │   │   return self._check_return(None, args[0])                                          │
│                                                                                                  │
│ /home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/IPython/core/formatters.py:708  │
│ in __call__                                                                                      │
│                                                                                                  │
│    705 │   │   │   │   singleton_pprinters=self.singleton_printers,                              │
│    706 │   │   │   │   type_pprinters=self.type_printers,                                        │
│    707 │   │   │   │   deferred_pprinters=self.deferred_printers)                                │
│ ❱  708 │   │   │   printer.pretty(obj)                                                           │
│    709 │   │   │   printer.flush()                                                               │
│    710 │   │   │   return stream.getvalue()                                                      │
│    711                                                                                           │
│                                                                                                  │
│ /home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/IPython/lib/pretty.py:410 in    │
│ pretty                                                                                           │
│                                                                                                  │
│   407 │   │   │   │   │   │   │   │   return meth(obj, self, cycle)                              │
│   408 │   │   │   │   │   │   if cls is not object \                                             │
│   409 │   │   │   │   │   │   │   │   and callable(cls.__dict__.get('__repr__')):                │
│ ❱ 410 │   │   │   │   │   │   │   return _repr_pprint(obj, self, cycle)                          │
│   411 │   │   │                                                                                  │
│   412 │   │   │   return _default_pprint(obj, self, cycle)                                       │
│   413 │   │   finally:                                                                           │
│                                                                                                  │
│ /home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/IPython/lib/pretty.py:778 in    │
│ _repr_pprint                                                                                     │
│                                                                                                  │
│   775 def _repr_pprint(obj, p, cycle):                                                           │
│   776 │   """A pprint that just redirects to the normal repr function."""                        │
│   777 │   # Find newlines and replace them with p.break_() 

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/IPython/core/formatters.py:223  │
│ in catch_format_error                                                                            │
│                                                                                                  │
│    220 def catch_format_error(method, self, *args, **kwargs):                                    │
│    221 │   """show traceback on failed format call"""                                            │
│    222 │   try:                                                                                  │
│ ❱  223 │   │   r = method(self, *args, **kwargs)                                                 │
│    224 │   except NotImplementedError:                                                           │
│    225 │   │   # don't warn on NotImplementedErrors                                              │
│    226 │   │   return self._check_return(None, args[0])                                          │
│                                                                                                  │
│ /home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/IPython/core/formatters.py:344  │
│ in __call__                                                                                      │
│                                                                                                  │
│    341 │   │   │   # Finally look for special method names                                       │
│    342 │   │   │   method = get_real_method(obj, self.print_method)                              │
│    343 │   │   │   if method is not None:                                                        │
│ ❱  344 │   │   │   │   return method()                                                           │
│    345 │   │   │   return None                                                                   │
│    346 │   │   else:                                                                             │
│    347 │   │   │   return None                                                                   │
│                                                                                                  │
│ /home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/nbdev/showdoc.py:168 in         │
│ _repr_markdown_                                                                                  │
│                                                                                                  │
│   165 │   "Markdown renderer for `show_doc`"                                                     │
│   166 │   def _repr_markdown_(self):                                                             │
│   167 │   │   doc = '---\n\n'                                                                    │
│ ❱ 168 │   │   src = NbdevLookup().code(self.fn)                                                  │
│   169 │   │   if src: doc += _ext_link(src, 'source', 'style="float:right; font-size:smaller"'   │
│   170 │   │   h = '#'*self.title_level                                                           │
│   171 │   │   doc += f'{h} {self.nm}\n\n'                                                        │
│                                                                                                  │
│ /home/quan/anaconda3/envs/fastai_v2/lib/python3.10/site-packages/nbdev/doclinks.py:200 in        │
│ __init__                                                                                         │
│                                                                                                  │
│   197 │   │   strip_libs = L(strip_libs)                                                         │
│   198 │   │   if incl_libs is not None: incl_libs = (L(incl_libs)+strip_libs).unique()           │
│   199 │   │   # Dict from lib name to _nbdev module for inc

In [ ]:
#| hide
#| export
def _forward_pass_for_predictions(batch,
                                 model=None, # NLP model
                                 topk=1, # Number of labels to return for each head
                                 use_softmax=True, # To whether use softmax or sigmoid at the last activation func
                                 device=None, # CPU or GPU
                                 tokenizer=None, # HuggingFace tokenizer
                                 data_collator=None, # HuggingFace data collator
                                 cols_to_remove=[], # list of keys (columns) to remove from ```batch```
                                 label_names=[], # Names of the label columns
                                 label_sizes=[], # Size of each label
                                 is_dhc=False
                                 ):
    if data_collator is not None:
# --- Convert from  
# {'input_ids': [tensor([    0, 10444,   244, 14585,   125,  2948,  5925,   368,     2]), 
#                tensor([    0, 16098,  2913,   244,   135,   198, 34629,  6356,     2])]
# 'attention_mask': [tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), 
#                    tensor([1, 1, 1, 1, 1, 1, 1, 1, 1])]
#                    }
# --- to
#         [{'input_ids': tensor([    0, 10444,   244, 14585,   125,  2948,  5925,   368,     2]),
#           'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1])},
#          {'input_ids': tensor([    0, 16098,  2913,   244,   135,   198, 34629,  6356,     2]),
#           'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1])}]

        # remove string text, due to transformer new version       
        collator_inp = []
        ks = [k for k in batch.keys() if k not in cols_to_remove]
        vs = [batch[k] for k in ks]
        for pair in zip(*vs):
            collator_inp.append({k:v for k,v in zip(ks,pair)})
        
        batch = data_collator(collator_inp)
        
    inputs = {k:v.to(device) for k,v in batch.items()
              if k in tokenizer.model_input_names}
    
    _f = partial(torch.nn.functional.softmax,dim=1) if use_softmax else torch.sigmoid
    
    with torch.no_grad():
        outputs = model(**inputs)
        outputs_logits = outputs.logits
        outputs_list=[]
        if is_dhc:
            # outputs_logits will be a list of n (typically 2) heads' logits, each has shape (bs,class_size)
            for i in range(len(label_names)):
                outputs_list.append(_f(outputs_logits[i].cpu()))
        else:
            # outputs_logits will have shape (bs,sum of all class sizes). We split into each class
            outputs_logits = outputs_logits.cpu()
            _s=0
            _e=label_sizes[0]
            for i in range(len(label_names)):
                outputs_list.append(_f(outputs_logits[:,_s:_e]))
                _s+=label_sizes[i]
                _e+=label_sizes[i+1] if i+1<len(label_names) else 0
        
        # save prediction and probability
        pred_label_list=[]
        pred_prob_list=[]
        for i in range(len(label_names)):
            _p,_l = torch.topk(outputs_list[i],topk,dim=-1)
            pred_label_list.append(_l)
            pred_prob_list.append(_p)
        if topk==1:
            for i in range(len(label_names)):
                pred_label_list[i]=pred_label_list[i][:,0]
                pred_prob_list[i]=pred_prob_list[i][:,0]
                
    results={}
    for i in range(len(label_names)):
        results[f'pred_{label_names[i]}']= pred_label_list[i].numpy()
        results[f'pred_prob_{label_names[i]}']= pred_prob_list[i].numpy()
    return results

In [ ]:
class ModelController():
    def __init__(self,model,data_store=None,metric_funcs=[accuracy_score],seed=42):
        self.model = model
        self.data_store = data_store
        self.metric_funcs = metric_funcs
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.seed = seed
        
    def fit(self,epochs,learning_rate,ddict=None,batch_size=16,weight_decay=0.01,lr_scheduler_type='cosine',
           o_dir='./tmp_weights',save_checkpoint=False,hf_report_to='none',
            compute_metrics=None,grad_accum_steps=2,tokenizer=None,data_collator=None):
        
        if tokenizer is None: tokenizer=check_and_get_attribute(self.data_store,'tokenizer')
        if data_collator is None: data_collator=getattr(self.data_store,'data_collator',None)
        if ddict is None: ddict = check_and_get_attribute(self.data_store,'main_ddict')
        
        if len(set(ddict.keys()) & set(['train','training']))==0:
            raise ValueError("Missing the following key for DatasetDict: train/training")
        no_valid= len(set(ddict.keys()) & set(['validation','val']))==0

        _compute_metrics = partial(compute_metrics,metric_funcs=self.metric_funcs)
        

        trainer = finetune(learning_rate,batch_size,weight_decay,epochs,
                           ddict,tokenizer,o_dir,
                         save_checkpoint=save_checkpoint,model=self.model,
                         data_collator=data_collator,compute_metrics=_compute_metrics,
                           grad_accum_steps=grad_accum_steps,
                         lr_scheduler_type=lr_scheduler_type,no_valid=no_valid,
                           seed=self.seed,report_to=hf_report_to)
        self.trainer = trainer
        
    def predict_raw_text(self,content,batch_size=1,use_softmax=True,topk=1):
        if not isinstance(self.data_store,text_store.TextDataMain) or not self.data_store._main_called:
            raise ValueError('This functionality needs a TextDataMain object which already processed some training data')
        with HiddenPrints():
            test_ddict = self.data_store.get_test_datasetdict_from_dict(content)
            df_result =  self.predict_ddict(ddict=test_ddict,ds_type='test',batch_size=batch_size,use_softmax=use_softmax,topk=topk)
        return df_result
    
    def predict_ddict(self,ddict=None,ds_type='test',batch_size=16,use_softmax=True,topk=1,
                      tokenizer=None,data_collator=None,
                      label_names=None,label_lists=None,is_dhc=False):
        
        if tokenizer is None: tokenizer=check_and_get_attribute(self.data_store,'tokenizer')
        if data_collator is None: data_collator=getattr(self.data_store,'data_collator',None)
        if label_names is None: label_names=check_and_get_attribute(self.data_store,'label_names')
        if label_lists is None: label_lists = check_and_get_attribute(self.data_store,'label_lists')
        if not isinstance(label_names,list):
            label_names=[label_names]
        if not isinstance(label_lists[0],list):
            label_lists=[label_lists]    
            
        label_sizes = [len(cs) for cs in label_lists]
        if ddict is None: ddict = check_and_get_attribute(self.data_store,'main_ddict') 
        if ds_type not in ddict.keys():
            raise ValueError(f'{ds_type} is not in the given DatasetDict keys')
        
        ddict.set_format("torch",
                        columns=["input_ids", "attention_mask"])
        print_msg('Start making predictions',20)
        preserved_kws=['input_ids', 'token_type_ids', 'attention_mask','label'] # hard-coded
        cols_to_remove = (set(ddict[ds_type].features.keys()) - set(preserved_kws)) | {'text'}
        ddict[ds_type] = ddict[ds_type].map(
            partial(_forward_pass_for_predictions,model=self.model,
                    topk=topk,
                    use_softmax=use_softmax,
                    device=self.device,
                    tokenizer=tokenizer,
                    data_collator=data_collator,
                    cols_to_remove=cols_to_remove,
                   label_names=label_names,
                    label_sizes=label_sizes,
                    is_dhc = is_dhc
                   ), 
            batched=True, batch_size=batch_size)
    
        ddict.set_format("pandas")
        
        df_result = ddict[ds_type][:]
        cols_to_keep = [c for c in df_result.columns.values if c not in preserved_kws[:-1]]
        df_result = df_result.loc[:,cols_to_keep]
        
        # convert pred id to string label
        for i in range(len(label_names)):        
            if topk==1:
                df_result[f'pred_{label_names[i]}'] = df_result[f'pred_{label_names[i]}'].apply(lambda x: label_lists[i][int(x)])
            else:
                df1 = pd.DataFrame(df_result[f'pred_{label_names[i]}'].to_list(),columns=[f'pred_{label_names[i]}_top{j}' for j in range(1,topk+1)])
                df1_prob = pd.DataFrame(df_result[f'pred_prob_{label_names[i]}'].to_list(),columns=[f'pred_prob_{label_names[i]}_top{j}' for j in range(1,topk+1)])
                
                for j in range(1,topk+1):
                    df1[f'pred_{label_names[i]}_top{j}'] =  df1[f'pred_{label_names[i]}_top{j}'].apply(lambda x: label_lists[i][int(x)])

                df_result = pd.concat([df_result,df1,df1_prob],axis=1)
                
        return df_result

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()